# Homework 4 — Evaluation
**LLM Zoomcamp 2026 | Cohort 2026**

Stack: Groq (llama-3.3-70b) · minsearch · ONNX embeddings · Hit Rate · MRR · RRF

Objetivo: responder la pregunta abierta de HW2 — ¿cuál método de búsqueda es mejor?
Medimos keyword search, vector search e hybrid search con las mismas 360 preguntas.

In [1]:
# ── Celda 1: Setup ───────────────────────────────────────────────────────────
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import List

load_dotenv(dotenv_path=Path("/home/juangraciano/Documentos/llm-zoomcamp/my-homework/01-agentic-rag/.env"))

# Cliente Groq — 100% compatible con OpenAI SDK
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY", ""),
    base_url="https://api.groq.com/openai/v1"
)
MODEL = "llama-3.3-70b-versatile"
print("Cliente Groq listo")

Cliente Groq listo


In [2]:
# ── Celda 2: Q1 — Generar ground truth para 3 páginas ───────────────────────
# evaluation_utils.py usa client.responses.parse() → solo OpenAI Responses API
# Groq usa chat.completions → implementamos nuestra propia versión con JSON mode

from gitsource import GithubRepositoryDataReader

# Modelo Pydantic para las 5 preguntas generadas por página
class Questions(BaseModel):
    questions: List[str]

def llm_structured(instructions: str, user_prompt: str) -> tuple:
    """Genera output estructurado usando Groq con JSON mode + Pydantic."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user",   "content": user_prompt}
        ],
        response_format={"type": "json_object"}  # Groq garantiza JSON válido
    )
    raw_json = json.loads(response.choices[0].message.content)
    parsed   = Questions(**raw_json)              # Validación con Pydantic
    return parsed, response.usage

# Instrucciones para generar preguntas de estudiante
DATA_GEN_INSTRUCTIONS = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.

Return a JSON object with a single key 'questions' containing a list of 5 strings.
""".strip()

# Cargamos los 72 documentos (igual que HW1 y HW2)
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Páginas cargadas: {len(documents)}")

Páginas cargadas: 72


In [3]:
# ── Celda 3: Q1 — Generar preguntas para las primeras 3 páginas ──────────────
# El HW pide las páginas 01-intro.md, 02-environment.md, 03-rag.md
target_pages = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

input_token_counts = []

for page in target_pages:
    # Buscamos el documento con ese filename
    doc = next(d for d in documents if d["filename"] == page)

    # Prompt de usuario: filename + contenido de la página
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})

    # Llamada al LLM con output estructurado
    questions_obj, usage = llm_structured(DATA_GEN_INSTRUCTIONS, user_prompt)

    # Groq usa prompt_tokens (no input_tokens como OpenAI Responses API)
    input_tokens = usage.prompt_tokens
    input_token_counts.append(input_tokens)

    print(f"  {page}")
    print(f"    Input tokens : {input_tokens}")
    print(f"    Preguntas    : {questions_obj.questions[0][:60]}...")
    print()

avg_input_tokens = sum(input_token_counts) / len(input_token_counts)
print(f"Q1 — Promedio de input tokens: {avg_input_tokens:.0f}")

  01-agentic-rag/lessons/01-intro.md
    Input tokens : 1051
    Preguntas    : What is the main goal of building a Retrieval-Augmented Gene...

  01-agentic-rag/lessons/02-environment.md
    Input tokens : 1324
    Preguntas    : What programming language and environment do I need to have ...

  01-agentic-rag/lessons/03-rag.md
    Input tokens : 1797
    Preguntas    : How can we create a system that answers student questions in...

Q1 — Promedio de input tokens: 1391


In [4]:
# ── Celda 4: Cargar ground truth completo ────────────────────────────────────
# 360 preguntas pre-generadas por el curso para todos los 72 documentos
ground_truth_df = pd.read_csv(
    "/home/juangraciano/Documentos/llm-zoomcamp/cohorts/2026/04-evaluation/ground-truth.csv"
)
ground_truth = ground_truth_df.to_dict(orient="records")

print(f"Total de preguntas en ground truth: {len(ground_truth)}")
print(f"\nEjemplo (primera pregunta):")
print(f"  question : {ground_truth[0]['question'][:80]}...")
print(f"  filename : {ground_truth[0]['filename']}")

Total de preguntas en ground truth: 360

Ejemplo (primera pregunta):
  question : What exactly is a retrieval-augmented generation system, and why does it help wi...
  filename : 01-agentic-rag/lessons/01-intro.md


In [6]:
# ── Celda 5: Construir índices de búsqueda (igual que HW2) ───────────────────
import numpy as np
from gitsource import chunk_documents
from minsearch import Index, VectorSearch
from embedder import Embedder

# Los modelos ONNX están en la carpeta de HW2
MODELS_PATH = "/home/juangraciano/Documentos/llm-zoomcamp/my-homework/02-vector-search/models"

# 295 chunks de 2000 chars con paso de 1000 (igual que HW2)
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total de chunks: {len(chunks)}")

# Índice de keyword search (TF-IDF)
keyword_index = Index(text_fields=["content"], keyword_fields=["filename"])
keyword_index.fit(chunks)
print("Índice keyword listo")

# Embedder ONNX (sin PyTorch)
embedder = Embedder(path=f"{MODELS_PATH}/Xenova/all-MiniLM-L6-v2")


# Matriz de embeddings (295, 384)
print("Generando embeddings de 295 chunks...")
X = embedder.encode_batch([c["content"] for c in chunks])
print(f"Matriz X: {X.shape}")

# Índice de vector search
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)
print("Índice vectorial listo")

Total de chunks: 295
Índice keyword listo
Generando embeddings de 295 chunks...
Matriz X: (295, 384)
Índice vectorial listo


In [7]:
# ── Celda 6: Funciones de búsqueda ───────────────────────────────────────────

def text_search(query: str, num_results: int = 5) -> list:
    """Keyword search sobre los 295 chunks."""
    return keyword_index.search(query, num_results=num_results)

def vector_search(query: str, num_results: int = 5) -> list:
    """Vector search semántico usando embeddings ONNX."""
    v = embedder.encode(query)
    return vector_index.search(v, num_results=num_results)

def rrf(result_lists: list, k: int = 60, num_results: int = 5) -> list:
    """Reciprocal Rank Fusion — combina múltiples listas de resultados."""
    scores = {}
    docs   = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key]   = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query: str, k: int = 60) -> list:
    """Hybrid search: fusiona keyword + vector con RRF."""
    text_results   = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

# Q2: primer resultado de text_search con la primera pregunta del ground truth
q = ground_truth[0]["question"]
print(f"Pregunta: {q[:80]}...")
print(f"Respuesta esperada (ground truth): {ground_truth[0]['filename']}")
print()

text_results = text_search(q)
print(f"Q2 — Primer resultado text_search  : {text_results[0]['filename']}")

vector_results = vector_search(q)
print(f"Q3 — Primer resultado vector_search: {vector_results[0]['filename']}")

Pregunta: What exactly is a retrieval-augmented generation system, and why does it help wi...
Respuesta esperada (ground truth): 01-agentic-rag/lessons/01-intro.md

Q2 — Primer resultado text_search  : 01-agentic-rag/lessons/03-rag.md
Q3 — Primer resultado vector_search: 01-agentic-rag/lessons/01-intro.md


In [8]:
# ── Celda 7: Funciones de evaluación ─────────────────────────────────────────
# Estas son las métricas que respondieron la pregunta abierta de HW2

def compute_relevance(search_fn, record: dict) -> list:
    """
    Ejecuta la búsqueda y devuelve lista de 0/1:
    1 si el chunk recuperado pertenece al filename correcto, 0 si no.
    """
    results = search_fn(record["question"])
    correct_filename = record["filename"]
    return [1 if r["filename"] == correct_filename else 0 for r in results]

def hit_rate(relevance_list: list) -> float:
    """
    Fracción de preguntas donde el documento correcto aparece en algún resultado.
    Hit Rate = P(el correcto aparece en top-5)
    """
    return sum(1 for rel in relevance_list if any(r == 1 for r in rel)) / len(relevance_list)

def mrr(relevance_list: list) -> float:
    """
    Mean Reciprocal Rank: premia encontrar el correcto en posiciones más altas.
    MRR = media de 1/rank del primer resultado correcto (0 si no aparece)
    """
    scores = []
    for rel in relevance_list:
        score = 0.0
        for rank, r in enumerate(rel, start=1):
            if r == 1:
                score = 1 / rank  # 1/1=1.0, 1/2=0.5, 1/3=0.33...
                break
        scores.append(score)
    return sum(scores) / len(scores)

def evaluate(search_fn, ground_truth: list) -> dict:
    """Evalúa una función de búsqueda sobre todo el ground truth."""
    relevance_list = [compute_relevance(search_fn, record) for record in ground_truth]
    return {
        "hit_rate": round(hit_rate(relevance_list), 4),
        "mrr":      round(mrr(relevance_list), 4)
    }

print("Funciones de evaluación definidas")

Funciones de evaluación definidas


In [9]:
# ── Celda 8: Q4 — Evaluar text_search ────────────────────────────────────────
print("Evaluando keyword search sobre 360 preguntas...")
text_metrics = evaluate(text_search, ground_truth)

print(f"\nQ4 — Text search:")
print(f"  Hit Rate : {text_metrics['hit_rate']}")
print(f"  MRR      : {text_metrics['mrr']}")

Evaluando keyword search sobre 360 preguntas...

Q4 — Text search:
  Hit Rate : 0.7583
  MRR      : 0.5943


In [10]:
# ── Celda 9: Q5 — Evaluar vector_search ──────────────────────────────────────
print("Evaluando vector search sobre 360 preguntas...")
vector_metrics = evaluate(vector_search, ground_truth)

print(f"\nQ5 — Vector search:")
print(f"  Hit Rate : {vector_metrics['hit_rate']}")
print(f"  MRR      : {vector_metrics['mrr']}")

Evaluando vector search sobre 360 preguntas...

Q5 — Vector search:
  Hit Rate : 0.725
  MRR      : 0.5486


In [11]:
# ── Celda 10: Q6 — Tuning del parámetro k en hybrid search ───────────────────
# k controla cuánto importa la posición en RRF:
# k pequeño → posición importa mucho (gap grande entre rank 1 y rank 2)
# k grande  → posición importa poco (todos los resultados pesan similar)

print("Evaluando hybrid search para diferentes valores de k...\n")

k_values = [1, 50, 100, 200]
results_k = {}

for k in k_values:
    # Función wrapper que fija el valor de k
    def hybrid_k(query, _k=k):
        return hybrid_search(query, k=_k)

    metrics = evaluate(hybrid_k, ground_truth)
    results_k[k] = metrics
    print(f"  k={k:3d} → Hit Rate: {metrics['hit_rate']:.4f} | MRR: {metrics['mrr']:.4f}")

# El mejor k es el que maximiza MRR (en caso de empate, el más pequeño)
best_k = min(results_k, key=lambda k: (-results_k[k]['mrr'], k))
print(f"\nQ6 — Mejor k para MRR: {best_k}")

Evaluando hybrid search para diferentes valores de k...

  k=  1 → Hit Rate: 0.8389 | MRR: 0.6482
  k= 50 → Hit Rate: 0.8361 | MRR: 0.6379
  k=100 → Hit Rate: 0.8361 | MRR: 0.6379
  k=200 → Hit Rate: 0.8361 | MRR: 0.6379

Q6 — Mejor k para MRR: 1


In [12]:
# ── Celda 11: Resumen final ───────────────────────────────────────────────────
print("=" * 55)
print("RESUMEN HW4 — ¿Qué método de búsqueda es mejor?")
print("=" * 55)
print(f"{'Método':<20} {'Hit Rate':>10} {'MRR':>10}")
print("-" * 45)
print(f"{'text_search':<20} {text_metrics['hit_rate']:>10.4f} {text_metrics['mrr']:>10.4f}")
print(f"{'vector_search':<20} {vector_metrics['hit_rate']:>10.4f} {vector_metrics['mrr']:>10.4f}")
print(f"{'hybrid (best k)':<20} {results_k[best_k]['hit_rate']:>10.4f} {results_k[best_k]['mrr']:>10.4f}")
print(f"\nMejor k para hybrid: {best_k}")

RESUMEN HW4 — ¿Qué método de búsqueda es mejor?
Método                 Hit Rate        MRR
---------------------------------------------
text_search              0.7583     0.5943
vector_search            0.7250     0.5486
hybrid (best k)          0.8389     0.6482

Mejor k para hybrid: 1
